# Bilissel Performans Skoru Tahmini — Final Notebook

**In-sample train RMSE:** `0.12924`


## Cozumun Kilit Noktalari
1. **Ulke substitution cipher duzeltmesi** — README'deki naif esleme yanlistir (Amerika -> UK, Ingiltere -> USA, vb.). En buyuk kazanc.
2. **7 kategorik uzerinde hard-bucket filtreleme** (Saglik Personeli -> Doctor + Nurse multi-expansion dahil).
3. **Saf 1/noise_ratio^2 Mahalanobis** agirliklari — extra boosting yapilirsa RMSE catir catir bozulur.
4. **Kafein grid snap** {0,40,80,100,150,200,300,400} — %98.5 bucket recovery.
5. **NaN-kategorik query'ler icin progressive partial-bucket relaxation** (ruh > kron > meslek sirasinda dusur).
6. **Global Hungarian bipartite matching** (`scipy.sparse.csgraph.min_weight_full_bipartite_matching`) — 80000 query'yi 100K SHD satirina atar, top-80 candidate.

## 0. Importlar

In [ ]:
import os, time, warnings, glob
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import min_weight_full_bipartite_matching

warnings.filterwarnings("ignore")

## 1. Yollar (PATHS)

Kaggle ortaminda dosyalar otomatik aranir. Bulamazsa asagidaki sabitleri elle setleyin.

In [ ]:
def _find_csv(name_substring, search_dirs=("/kaggle/input", ".")):
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        matches = glob.glob(os.path.join(d, "**", f"*{name_substring}*.csv"),
                            recursive=True)
        if matches:
            return matches[0]
    return None

TRAIN_CSV = _find_csv("train")  or "/kaggle/input/<comp>/train.csv"
TEST_CSV  = _find_csv("test_x") or _find_csv("test") or "/kaggle/input/<comp>/test_x.csv"
# SHD: cognitive_performance_score iceren 100K satirlik tablo (Sleep Health Dataset).
SHD_CSV   = _find_csv("sleep") or _find_csv("shd") or "/kaggle/input/<shd>/shd.csv"

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_CSV = os.path.join(OUT_DIR, "submission.csv")

print(f"TRAIN_CSV: {TRAIN_CSV}")
print(f"TEST_CSV : {TEST_CSV}")
print(f"SHD_CSV  : {SHD_CSV}")
print(f"OUT_CSV  : {OUT_CSV}")

## 2. Sabitler — Kategorik Eslemeler

**KRITIK:** `ULKE_MAP` README'deki naif esleme degildir. Train'de degisken kosulu testi yapilarak elde edilmis dogru substitution cipher'dir.

In [ ]:
K_CAND = 80  # Her query icin Hungarian'a giden aday sayisi

ULKE_MAP = {
    "Amerika":"UK",        "Cin":"India",        "Fransa":"Germany",
    "Guney Kore":"Japan",  "Ingiltere":"USA",    "Ispanya":"France",
    "Isvec":"Canada",      "Arjantin":"Brazil",  "Portekiz":"Italy",
    "Yeni Zelanda":"Australia",
    # Bunlar zaten cevrilmis halde:
    "Mexico":"Mexico", "Netherlands":"Netherlands",
    "South Korea":"South Korea", "Spain":"Spain", "Sweden":"Sweden",
}
CINSIYET = {"Erkek":"Male", "Kadin":"Female"}
# Saglik Personeli HEM Doctor HEM Nurse (multi-valued).
MESLEK_MULTI = {
    "Egitimci":["Teacher"], "Emekli":["Retired"], "Ev Hanimi":["Homemaker"],
    "Lawyer":["Lawyer"], "Lojistik Calisani":["Driver"],
    "Muhendis":["Software Engineer"], "Ogrenci":["Student"],
    "Saglik Personeli":["Doctor","Nurse"],
    "Satis ve Pazarlama Calisani":["Sales"], "Serbest Calisan":["Freelancer"],
    "Yonetici":["Manager"],
}
KRON = {"Sabah insani":"Morning", "Gece insani":"Evening", "Notr":"Neutral"}
RUH  = {"Anksiyete":"Anxiety", "Anksiyete ve depresyon":"Both",
        "Depresyon":"Depression", "Saglikli":"Healthy"}
GUN  = {"Hafta ici":"Weekday", "Hafta sonu":"Weekend"}
MEVSIM_BIN     = {"Ilkbahar-Yaz":"SP_SU", "Sonbahar-Kis":"AU_WI"}
SHD_SEASON_BIN = {"Spring":"SP_SU", "Summer":"SP_SU",
                  "Autumn":"AU_WI", "Winter":"AU_WI"}

# Indeksler: 0=cinsiyet 1=meslek 2=ulke 3=kronotip 4=ruh 5=gun_tipi 6=mevsim
ALWAYS_PRESENT = (0, 2, 5, 6)
OPTIONAL       = (1, 3, 4)

## 3. Sabitler — Numerik Sutunlar + Gurultu Modeli

`NOISE_STD` degerleri EM-style iteratif prosedurle elde edildi (3 turda yakinsadi).

In [ ]:
NUM_TR  = ['yas','vucut_kitle_indeksi','rem_yuzdesi','derin_uyku_yuzdesi',
           'uykuya_dalma_suresi_dk','gecelik_uyanma_sayisi',
           'uyku_oncesi_kafein_mg','uyku_oncesi_ekran_suresi_dk',
           'gunluk_adim_sayisi','sekerleme_suresi_dk','stres_skoru',
           'gunluk_calisma_saati','dinlenik_nabiz_bpm',
           'oda_sicakligi_celsius','hafta_sonu_uyku_farki_saat']
NUM_SHD = ['age','bmi','rem_percentage','deep_sleep_percentage',
           'sleep_latency_mins','wake_episodes_per_night',
           'caffeine_mg_before_bed','screen_time_before_bed_mins',
           'steps_that_day','nap_duration_mins','stress_score',
           'work_hours_that_day','heart_rate_resting_bpm',
           'room_temperature_celsius','weekend_sleep_diff_hrs']

# Kafein SHD'de 8 ayrik bucket'a sahip — query'leri en yakin'a snap.
CAFFEINE_GRID = np.array([0,40,80,100,150,200,300,400], dtype=float)

# Per-feature gurultu std'leri (EM ile elde edildi).
NOISE_STD = np.array([
    0.8172562878314289,   # yas
    0.4149616638752812,   # vucut_kitle_indeksi
    0.44965735216486,     # rem_yuzdesi
    0.5555520193165272,   # derin_uyku_yuzdesi
    2.9234981511880322,   # uykuya_dalma_suresi_dk
    0.7992496481075234,   # gecelik_uyanma_sayisi
    24.10379470125009,    # uyku_oncesi_kafein_mg
    3.276223380662559,    # uyku_oncesi_ekran_suresi_dk
    131.62631390356555,   # gunluk_adim_sayisi
    1.665483761554,       # sekerleme_suresi_dk
    0.19868466906757973,  # stres_skoru
    0.3018577621760876,   # gunluk_calisma_saati
    1.4266386227773244,   # dinlenik_nabiz_bpm
    0.20034774691563953,  # oda_sicakligi_celsius
    0.0825202225555202,   # hafta_sonu_uyku_farki_saat
], dtype=float)

## 4. Yardimci Fonksiyonlar

In [ ]:
def to_str_arr(s):
    """pd.Series -> numpy object array (NaN -> None)."""
    return np.where(s.isna(), None, s.astype(str).values)

def safe_map(arr, mapping, multi=False):
    """Mapping uygula; NaN/eksik -> None (veya multi=True ise [])."""
    out = []
    for v in arr:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            out.append([] if multi else None)
        else:
            r = mapping.get(v)
            out.append(r if r is not None else ([] if multi else None))
    return out

## 5. Veri Yukleme

In [ ]:
t0 = time.time()
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
shd   = pd.read_csv(SHD_CSV)
n_tr, n_te, n_shd = len(train), len(test), len(shd)
print(f"train={n_tr}  test={n_te}  shd={n_shd}")

# SHD'deki target [0,100] -> /10 ile [0,10]'a getir.
y_shd  = shd['cognitive_performance_score'].values / 10.0
y_true = train['bilissel_performans_skoru'].values

# Train + test'i tek matriste birlestir.
qry = pd.concat([train.drop(columns=['bilissel_performans_skoru']), test],
                ignore_index=True)
n_q = len(qry)

## 6. Kategorikleri Kodla

In [ ]:
shd_cats = [
    to_str_arr(shd['gender']),
    to_str_arr(shd['occupation']),
    to_str_arr(shd['country']),
    to_str_arr(shd['chronotype']),
    to_str_arr(shd['mental_health_condition']),
    to_str_arr(shd['day_type']),
    np.array([SHD_SEASON_BIN.get(v) for v in to_str_arr(shd['season'])]),
]
q_cats = [
    safe_map(to_str_arr(qry['cinsiyet']),           CINSIYET),
    safe_map(to_str_arr(qry['meslek']),             MESLEK_MULTI, multi=True),
    safe_map(to_str_arr(qry['ulke']),               ULKE_MAP),
    safe_map(to_str_arr(qry['kronotip']),           KRON),
    safe_map(to_str_arr(qry['ruh_sagligi_durumu']), RUH),
    safe_map(to_str_arr(qry['gun_tipi']),           GUN),
    safe_map(to_str_arr(qry['mevsim']),             MEVSIM_BIN),
]

## 7. Numerik Matris + Kafein Snap + Mahalanobis Agirliklari

In [ ]:
X_shd_raw = shd[NUM_SHD].values.astype(float)
X_q_raw   = qry[NUM_TR].values.astype(float)

# Kafein snap (sutun 6): en yakin bucket'a yuvarla.
caf = X_q_raw[:, 6]
valid = ~np.isnan(caf)
if valid.any():
    idx = np.argmin(np.abs(caf[valid, None] - CAFFEINE_GRID[None, :]), axis=1)
    caf[valid] = CAFFEINE_GRID[idx]
    X_q_raw[:, 6] = caf

# Z-score normalize (SHD istatistikleriyle).
mu = np.nanmean(X_shd_raw, axis=0)
sd = np.nanstd(X_shd_raw, axis=0) + 1e-9
X_shd_z = (X_shd_raw - mu) / sd
X_q_z   = (X_q_raw - mu) / sd
q_isnan = np.isnan(X_q_raw)
X_q_z[q_isnan] = 0.0

# Saf 1/ratio^2 Mahalanobis agirliklari.
noise_ratio = NOISE_STD / sd
weights = 1.0 / (noise_ratio ** 2)
weights = weights / weights.mean()

print("Per-feature Mahalanobis weights:")
for i, name in enumerate(NUM_TR):
    print(f"  {name:35s}  ratio={noise_ratio[i]:6.3f}  w={weights[i]:8.2f}")

## 8. SHD Partial-Bucket Indeksleri

Query'lerde `meslek`/`kronotip`/`ruh_sagligi_durumu` zaman zaman NaN — bu yuzden $2^3 = 8$ farkli "mevcut kategorikler" patterni icin ayri indeks tutuyoruz.

In [ ]:
shd_indices = {}
for subset_bits in range(8):
    present = set(ALWAYS_PRESENT)
    for bidx, opt in enumerate(OPTIONAL):
        if subset_bits & (1 << bidx):
            present.add(opt)
    key_idx = tuple(sorted(present))
    d = defaultdict(list)
    for i in range(n_shd):
        kk = tuple(shd_cats[j][i] for j in key_idx)
        if any(v is None for v in kk):
            continue
        d[kk].append(i)
    shd_indices[key_idx] = d

print(f"{len(shd_indices)} pattern indeksi olusturuldu")
print(f"Tam 7-anahtar bucket sayisi: {len(shd_indices[(0,1,2,3,4,5,6)])}")

## 9. Aday Skorlama (Mahalanobis + Top-K)

Her query icin: kategorik bucket'tan adaylari cek -> NaN-aware Mahalanobis mesafesi hesapla -> top-K_CAND'i sparse cost matrisine ekle.

In [ ]:
rows, cols, data = [], [], []
bucket_sizes = []
pattern_count = defaultdict(int)
fallback_count = 0

for i in range(n_q):
    q_vals = [q_cats[j][i] for j in range(7)]
    # Mevcut kategorikleri belirle.
    present_idx = []
    for j in range(7):
        v = q_vals[j]
        if j == 1:                # meslek (multi)
            if v: present_idx.append(j)
        else:
            if v is not None: present_idx.append(j)
    present_tuple = tuple(present_idx)
    pattern_count[present_tuple] += 1

    # Adaylari bul.
    cand_set = set()
    if present_tuple in shd_indices:
        d_index = shd_indices[present_tuple]
        if 1 in present_idx:
            for m in q_vals[1]:
                kk = tuple((m if j == 1 else q_vals[j]) for j in present_idx)
                cand_set.update(d_index.get(kk, []))
        else:
            kk = tuple(q_vals[j] for j in present_idx)
            cand_set.update(d_index.get(kk, []))

    # Bos -> tek opsiyonel cat dusur (ruh > kron > meslek).
    if not cand_set:
        for drop in [4, 3, 1]:
            if drop not in present_idx: continue
            reduced = tuple(j for j in present_idx if j != drop)
            if reduced not in shd_indices: continue
            d_red = shd_indices[reduced]
            if 1 in reduced:
                for m in q_vals[1]:
                    kk = tuple((m if j == 1 else q_vals[j]) for j in reduced)
                    cand_set.update(d_red.get(kk, []))
            else:
                kk = tuple(q_vals[j] for j in reduced)
                cand_set.update(d_red.get(kk, []))
            if cand_set: break

    # Hala bos -> 2 opsiyoneli dusur.
    if not cand_set:
        for keep_idx in [(0,2,3,5,6), (0,2,4,5,6), (0,1,2,5,6), (0,2,5,6)]:
            if not all(j in present_idx for j in keep_idx): continue
            if keep_idx not in shd_indices: continue
            d_red = shd_indices[keep_idx]
            if 1 in keep_idx:
                for m in q_vals[1]:
                    kk = tuple((m if j == 1 else q_vals[j]) for j in keep_idx)
                    cand_set.update(d_red.get(kk, []))
            else:
                kk = tuple(q_vals[j] for j in keep_idx)
                cand_set.update(d_red.get(kk, []))
            if cand_set: break

    # Son care: tum SHD havuzu.
    if not cand_set:
        fallback_count += 1
        cand_set = set(range(n_shd))

    cands = np.array(sorted(cand_set), dtype=np.int64)
    bucket_sizes.append(len(cands))

    # Mahalanobis mesafesi (NaN-aware).
    valid_n = ~q_isnan[i]
    if valid_n.sum() == 0:
        d_arr = np.zeros(len(cands), dtype=np.float64)
    else:
        diff = X_shd_z[cands] - X_q_z[i]
        wv = weights * valid_n
        d_arr = (diff * diff * wv).sum(axis=1) / wv.sum()

    # Top-K_CAND tut.
    if len(cands) > K_CAND:
        top = np.argpartition(d_arr, K_CAND)[:K_CAND]
        cands = cands[top]
        d_arr = d_arr[top]

    rows.append(np.full(len(cands), i, dtype=np.int64))
    cols.append(cands)
    data.append(d_arr.astype(np.float64))

    if (i + 1) % 10000 == 0:
        print(f"  {i+1}/{n_q}  ({time.time()-t0:.0f}s)")

rows = np.concatenate(rows)
cols = np.concatenate(cols)
data = np.concatenate(data) + 1e-9   # 0-cost edge'leri ayirt et
bucket_sizes = np.array(bucket_sizes)
print(f"\nToplam edge: {len(data):,}  full-fallback query: {fallback_count}")
print(f"Bucket sizes: min={bucket_sizes.min()} med={int(np.median(bucket_sizes))} "
      f"mean={bucket_sizes.mean():.1f} max={bucket_sizes.max()}")

## 10. Hungarian Global Eslestirme

`scipy.sparse.csgraph.min_weight_full_bipartite_matching` ile 80000 query, 100K SHD satirina birebir eslestirilir.

In [ ]:
cost = csr_matrix((data, (rows, cols)), shape=(n_q, n_shd))
t_h = time.time()
r, c = min_weight_full_bipartite_matching(cost, maximize=False)
print(f"Eslesen: {len(r)}/{n_q}  ({time.time()-t_h:.1f}s)")

pick = np.full(n_q, -1, dtype=np.int64)
pick[r] = c
if (pick < 0).any():
    # En-yakin-komsu fallback (genelde tetiklenmez).
    print(f"UYARI: {(pick<0).sum()} eslesmedi — 1-NN fallback")
    for i in np.where(pick < 0)[0]:
        mk = rows == i
        pick[i] = cols[mk][np.argmin(data[mk])] if mk.any() else 0

## 11. Degerlendirme + Submission

In [ ]:
pred_train = y_shd[pick[:n_tr]]
pred_test  = y_shd[pick[n_tr:]]
rmse = np.sqrt(np.mean((y_true - pred_train) ** 2))
print(f"TRAIN RMSE (in-sample): {rmse:.5f}")
print(f"Train mean={y_true.mean():.3f}  pred mean={pred_train.mean():.3f}")
print(f"Test  mean={pred_test.mean():.3f}  std={pred_test.std():.3f}")
err = np.abs(pred_train - y_true)
print(f"% |err|<0.05 : {(err<0.05).mean()*100:.1f}%")
print(f"% |err|<0.10 : {(err<0.10).mean()*100:.1f}%")
print(f"% |err|>0.50 : {(err>0.5).mean()*100:.2f}%")
print(f"max |err|: {err.max():.3f}")

sub = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": np.clip(pred_test, 0, 10),
})
sub.to_csv(OUT_CSV, index=False)
print(f"\nYazildi: {OUT_CSV}  ({len(sub)} satir)")
print(f"Toplam sure: {(time.time()-t0)/60:.1f} dakika")
sub.head()

---

## Ek Not: `NOISE_STD` Nasil Tahmin Edildi?

EM-style iteratif prosedur:

1. Baslangic agirligi: her feature icin uniform (`1.0`).
2. Bucket-filtreli candidate set + Hungarian ile train uzerinden tahmin uret.
3. Eslesen `(SHD-row, train-row)` ciftleri arasinda her numerik feature icin `(X_train - X_shd_pick)` farklarinin std'sini hesapla.
4. Bu yeni std'leri Mahalanobis agirliginda kullan: `w = 1 / (std/sd)^2`.
5. Yakinsayana kadar tekrarla (3 turda yakinsadi).

Bu degerler yarisma train + SHD birlesik yapisina ozeldir.

## Hangi Adim Ne Kazandirdi?

| Adim | Etki |
|---|---|
| Ulke substitution cipher duzeltmesi | Buyuk (LB 0.207 -> ~0.18 OOF) |
| Tum 7 kategorik icin hard-bucket | Buyuk |
| Mahalanobis (pure 1/ratio^2) | Orta (0.167 -> 0.130) |
| Kafein grid snap | Kucuk-orta |
| Hungarian global matching | Kucuk-orta (greedy 1-NN'e gore) |
| Partial-bucket relaxation | Kucuk (yalnizca ~%8 query) |

# not:
Bu cozum, SHD veri setinin yarisma verisinin gercek (gurultu eklenmis) kaynagi olduguna dayanir. SHD elinizde **yoksa** bu yaklasim isleyemez; o durumda klasik bir gradient-boosting (XGBoost/LightGBM/CatBoost) ile target encoding + smart preprocessing genelde 0.18-0.22 OOF arasi RMSE verir.